# LSTM Fake Review Detection — Yelp — Supervised Contrastive Learning

**Model:** Bidirectional LSTM (128 units) + GloVe 300d + GlobalMaxPooling1D

**Method:** Two-stage supervised contrastive learning (SupCon)
- **Stage 1 — SupCon pretraining:** GloVe embedding frozen; encoder LSTM + projection head trained with supervised contrastive loss (temperature=0.15) on class-balanced batches (50% genuine / 50% fake). Light word-dropout augmentation (7%) applied to create diverse views.
- **Stage 2 — Classifier fine-tuning:** Projection head discarded. GloVe embedding unfrozen. Classifier head attached to encoder and trained on the full imbalanced training set with balanced batches; early stopping + LR reduction on validation macro F1.

**Training data:** Full unbalanced Yelp dataset from `yelp_split_imbalanced.csv` (~13% fake, after cross-dataset deduplication)

**Tuned for maximum macro F1:**
- **`MAX_LEN = 250` (not 150):** ~21% of Yelp reviews exceed 150 words; 250 cuts truncation to ~8% — the single biggest accuracy lever. *(Serving + eval code updated to 250 to match.)*
- **GloVe 300d (not 100d):** richer pre-trained embeddings.

*(Matched to the baseline and focal LSTM notebooks so the three-way ablation is strictly comparable.)*

**Architecture:**
- Encoder: Embedding(300d GloVe) → SpatialDropout1D(0.3) → BiLSTM(128) → GlobalMaxPooling1D → **256-dim**
- Projection head (Stage 1 only): 256 → Dense(256, relu) → Dense(128) → L2-normalize → **128-dim unit sphere**
- Classifier head (Stage 2): 256 → Dropout(0.5) → Dense(1, sigmoid)

---
### Instructions
1. Go to **Runtime → Change runtime type → A100 GPU** (or T4) before running.
2. Run `prepare_yelp_split_imbalanced.ipynb` first to generate `yelp_split_imbalanced.csv` on Google Drive.
3. Update file paths in **cell 4** if needed.
4. Run all cells in order.
5. Download `yelp_fake_lstm_contrastive_onnx.zip`, extract it, and place the folder at `data/yelp_fake_lstm_contrastive_onnx/`.

In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
!pip install -q tf2onnx onnxruntime

In [ ]:
import os
if not os.path.exists('glove.6B.300d.txt'):
    !wget -q --show-progress http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip glove.6B.300d.txt
    print('GloVe 300d downloaded.')
else:
    print('GloVe 300d already present.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THESE PATHS if your files are in a different location ──
SPLIT_CSV    = '/content/drive/MyDrive/yelp_split_imbalanced.csv'
BALANCED_CSV = '/content/drive/MyDrive/yelp_split.csv'

In [ ]:
import re, json, random, shutil
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Dense,
    Dropout, SpatialDropout1D, GlobalMaxPooling1D, Lambda,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ── Encoder ──────────────────────────────────────────────────────
# MAX_LEN 250, not 150: ~21% of Yelp reviews exceed 150 words; 250 cuts that
# to ~8%. NOTE: serving (score_reviews.py YELP_LSTM_MAX_LEN) and the eval
# scripts must use the SAME value — they have been updated to 250.
MAX_LEN       = 250
VOCAB_SIZE    = 50_000
EMBED_DIM     = 300        # GloVe 300d (richer than 100d) — must match glove.6B.300d.txt
LSTM_UNITS    = 128        # BiLSTM → 256-dim pooled output
GLOVE_PATH    = 'glove.6B.300d.txt'

# ── Projection head (Stage 1 only) ───────────────────────────────
PROJ_DIM_1    = 256        # 256 → 256 (wider than original 128)
PROJ_DIM_2    = 128        # 256 → 128  (L2 normalised; more angular room on hypersphere)

# ── SupCon stage ─────────────────────────────────────────────────
SUPCON_TEMP       = 0.15   # 0.07 → 0.15: smoother gradients for weak text signal
BATCH_SUPCON      = 256    # 128 genuine + 128 fake
EPOCHS_SUPCON     = 10     # 5 → 10: more time for Stage 1 convergence
LR_SUPCON         = 1e-3
AUGMENT_DROP_RATE = 0.07   # 7% word dropout during SupCon only

# ── Classifier stage ─────────────────────────────────────────────
BATCH_CLF     = 256        # 128 genuine + 128 fake (balanced batches)
EPOCHS_CLF    = 15
LR_CLF        = 2e-4       # 5e-4 → 2e-4: embedding is now pretrained (GloVe), lower LR protects it
PATIENCE_CLF  = 3

In [ ]:
def normalize_text(t):
    return re.sub(r'\s+', ' ', t.strip().lower())


def augment_batch(batch, drop_rate=0.07):
    """Random word dropout — zero out integer token IDs with probability drop_rate.

    Keeps integer dtype so token indices are never converted to float during
    augmentation. A True/False mask is applied via integer multiplication.
    """
    mask = np.random.random(batch.shape) > drop_rate   # bool array
    return (batch * mask).astype(batch.dtype)           # preserves int32


def supcon_loss(embeddings, labels, temperature=0.07):
    """
    Supervised Contrastive Loss (Khosla et al., 2020).
    embeddings : (N, D) float32 — L2-normalised projected embeddings
    labels     : (N,)  int32   — class labels
    """
    embeddings = tf.cast(embeddings, tf.float32)
    labels     = tf.cast(labels, tf.int32)
    N = tf.shape(embeddings)[0]

    sim = tf.matmul(embeddings, embeddings, transpose_b=True) / temperature

    # Numerical stability
    sim_max = tf.stop_gradient(tf.reduce_max(sim, axis=1, keepdims=True))
    sim     = sim - sim_max

    mask_self = tf.eye(N, dtype=tf.float32)
    labels_col = tf.reshape(labels, [-1, 1])
    labels_row = tf.reshape(labels, [1, -1])
    mask_pos   = tf.cast(tf.equal(labels_col, labels_row), tf.float32) - mask_self

    exp_sim   = tf.exp(sim) * (1.0 - mask_self)
    log_denom = tf.math.log(tf.reduce_sum(exp_sim, axis=1, keepdims=True) + 1e-9)
    log_prob  = sim - log_denom

    n_pos            = tf.reduce_sum(mask_pos, axis=1)
    mean_log_prob    = tf.reduce_sum(mask_pos * log_prob, axis=1) / (n_pos + 1e-9)
    valid            = tf.cast(n_pos > 0, tf.float32)
    loss             = -tf.reduce_sum(valid * mean_log_prob) / (tf.reduce_sum(valid) + 1e-9)
    return loss

In [ ]:
print('Loading yelp_split_imbalanced.csv ...')
df_split = pd.read_csv(SPLIT_CSV)
df_split['text_norm'] = df_split['text'].apply(normalize_text)

X_train = df_split[df_split['split'] == 'train']['text'].tolist()
y_train = df_split[df_split['split'] == 'train']['label'].tolist()
X_val   = df_split[df_split['split'] == 'val']['text'].tolist()
y_val   = df_split[df_split['split'] == 'val']['label'].tolist()
X_test  = df_split[df_split['split'] == 'test']['text'].tolist()
y_test  = df_split[df_split['split'] == 'test']['label'].tolist()

print('\nSplit counts:')
for name, X, y in [('Train', X_train, y_train), ('Val', X_val, y_val), ('Test', X_test, y_test)]:
    n_g = sum(1 for v in y if v == 0)
    n_f = sum(1 for v in y if v == 1)
    print(f'  {name:<6}: {len(X):>8,}  genuine={n_g:,}  fake={n_f:,}  fake%={n_f/len(y):.1%}')

# ── Leakage check ────────────────────────────────────────────────
# Balanced test texts must not appear in the imbalanced train OR val splits.
# (Both focal and contrastive training use yelp_split_imbalanced.csv, so one
#  check covers all ablation models.)
print('\nLeakage check ...')
bal_df = pd.read_csv(BALANCED_CSV)
bal_df['text_norm'] = bal_df['text'].apply(normalize_text)

pinned_test = set(bal_df[bal_df['split'] == 'test']['text_norm'])
train_norms = set(df_split[df_split['split'] == 'train']['text_norm'])
val_norms   = set(df_split[df_split['split'] == 'val']['text_norm'])

leaked_train = pinned_test & train_norms
leaked_val   = pinned_test & val_norms

print(f'  Balanced test texts : {len(pinned_test):,}')
print(f'  Overlap with train  : {len(leaked_train):,}')
print(f'  Overlap with val    : {len(leaked_val):,}')

if leaked_train:
    raise ValueError(
        f'LEAKAGE: {len(leaked_train):,} balanced test texts found in training set. '
        'Regenerate yelp_split_imbalanced.csv.'
    )
if leaked_val:
    raise ValueError(
        f'LEAKAGE: {len(leaked_val):,} balanced test texts found in validation set. '
        'Regenerate yelp_split_imbalanced.csv.'
    )
print('  PASS — No leakage.')

In [ ]:
print('Building tokenizer on training set ...')
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)
print(f'  Vocabulary: {len(tokenizer.word_index):,} unique words')

def encode(texts):
    return pad_sequences(
        tokenizer.texts_to_sequences(texts),
        maxlen=MAX_LEN, padding='post', truncating='post',
    )

X_train_seq = encode(X_train)
X_val_seq   = encode(X_val)
X_test_seq  = encode(X_test)
y_train_arr = np.array(y_train, dtype=np.float32)
y_val_arr   = np.array(y_val,   dtype=np.float32)
y_test_arr  = np.array(y_test,  dtype=np.float32)
print(f'  X_train_seq: {X_train_seq.shape}  X_val_seq: {X_val_seq.shape}')

# ── GloVe ────────────────────────────────────────────────────────
print('Loading GloVe ...')
glove = {}
with open(GLOVE_PATH, encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        glove[parts[0]] = np.array(parts[1:], dtype=np.float32)
print(f'  {len(glove):,} vectors loaded')

embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM), dtype=np.float32)
hits, misses = 0, 0
for word, idx in tokenizer.word_index.items():
    if idx >= VOCAB_SIZE:
        continue
    vec = glove.get(word)
    if vec is not None:
        embedding_matrix[idx] = vec
        hits += 1
    else:
        misses += 1
print(f'  GloVe coverage: {hits/(hits+misses):.1%}  ({hits:,} hits / {misses:,} misses)')

In [ ]:
for path in [
    'yelp_fake_lstm_contrastive.keras',
    'yelp_fake_lstm_contrastive_savedmodel',
    'yelp_fake_lstm_contrastive.onnx',
    'yelp_fake_lstm_contrastive_tokenizer.json',
    'yelp_fake_lstm_contrastive_threshold.json',
    'yelp_fake_lstm_contrastive_metrics.csv',
    'yelp_fake_lstm_contrastive_config.json',
    'yelp_fake_lstm_contrastive_onnx.zip',
]:
    if os.path.isdir(path):
        shutil.rmtree(path); print(f'Removed dir : {path}')
    elif os.path.isfile(path):
        os.remove(path);     print(f'Removed file: {path}')
print('Clean.')

In [ ]:
# ── Encoder ──────────────────────────────────────────────────────
inputs  = Input(shape=(MAX_LEN,), name='input_layer')
x       = Embedding(VOCAB_SIZE, EMBED_DIM, weights=[embedding_matrix], trainable=True)(inputs)
x       = SpatialDropout1D(0.3)(x)
x       = Bidirectional(LSTM(LSTM_UNITS, return_sequences=True, kernel_regularizer=l2(1e-4)))(x)
enc_out = GlobalMaxPooling1D()(x)   # (batch, 256)
encoder = Model(inputs, enc_out, name='encoder')

# ── Projection head — Stage 1 only ───────────────────────────────
proj = Dense(PROJ_DIM_1, activation='relu')(encoder.output)
proj = Dense(PROJ_DIM_2)(proj)
proj = Lambda(lambda t: tf.math.l2_normalize(t, axis=1), name='l2_norm')(proj)
encoder_proj = Model(encoder.input, proj, name='encoder_proj')

# ── Classifier head — Stage 2 ─────────────────────────────────────
clf_out  = Dropout(0.5)(encoder.output)
clf_out  = Dense(1, activation='sigmoid', name='output_0')(clf_out)
classifier = Model(encoder.input, clf_out, name='classifier')

print(f'Encoder output      : {encoder.output_shape}')
print(f'Projection output   : {encoder_proj.output_shape}')
print(f'Classifier output   : {classifier.output_shape}')
encoder.summary()

In [ ]:
def balanced_batch_gen(X, y, batch_size, augment=True, drop_rate=0.07):
    """
    Infinite generator yielding batches with exactly batch_size//2 genuine
    and batch_size//2 fake samples. Applies word-dropout augmentation if requested.
    Token IDs are kept as int32 throughout — no float conversion.
    """
    X, y   = np.array(X), np.array(y)
    idx_g  = np.where(y == 0)[0]
    idx_f  = np.where(y == 1)[0]
    half   = batch_size // 2
    while True:
        g = np.random.choice(idx_g, half, replace=len(idx_g) < half)
        f = np.random.choice(idx_f, half, replace=len(idx_f) < half)
        idx = np.concatenate([g, f])
        np.random.shuffle(idx)
        bX = X[idx]                  # int32 from pad_sequences — preserve dtype
        by = y[idx].astype(np.int32)
        if augment:
            bX = augment_batch(bX, drop_rate)
        yield bX, by

In [ ]:
print('=' * 60)
print('Stage 1 — Supervised Contrastive Pretraining')
print('=' * 60)

# Freeze GloVe embedding: SupCon gradient signal on Yelp text is weak.
# Keeping GloVe vectors frozen prevents them from drifting under noisy
# contrastive gradients. BiLSTM + GlobalMaxPool + projection are still updated.
for layer in encoder.layers:
    if isinstance(layer, tf.keras.layers.Embedding):
        layer.trainable = False
        print(f'  Embedding frozen ({layer.name}) — LSTM + projection will be updated')
        break

optimizer_supcon = Adam(learning_rate=LR_SUPCON)
steps_per_epoch  = len(X_train) // BATCH_SUPCON
gen              = balanced_batch_gen(X_train_seq, y_train_arr, BATCH_SUPCON,
                                      augment=True, drop_rate=AUGMENT_DROP_RATE)
supcon_history   = []

for epoch in range(1, EPOCHS_SUPCON + 1):
    epoch_losses = []
    for step in range(steps_per_epoch):
        bX, by = next(gen)
        bX_t   = tf.constant(bX)
        by_t   = tf.constant(by)
        with tf.GradientTape() as tape:
            projections = encoder_proj(bX_t, training=True)
            loss        = supcon_loss(projections, by_t, temperature=SUPCON_TEMP)
        grads = tape.gradient(loss, encoder_proj.trainable_variables)
        optimizer_supcon.apply_gradients(zip(grads, encoder_proj.trainable_variables))
        epoch_losses.append(float(loss))
        if (step + 1) % 500 == 0:
            print(f'  Epoch {epoch} | Step {step+1}/{steps_per_epoch} | Loss: {np.mean(epoch_losses):.4f}')
    mean_loss = np.mean(epoch_losses)
    supcon_history.append(mean_loss)
    print(f'Epoch {epoch}/{EPOCHS_SUPCON} — SupCon Loss: {mean_loss:.4f}\n')

print('Stage 1 complete. Encoder weights (excl. embedding) updated by SupCon objective.')

In [ ]:
print('=' * 60)
print('Stage 2 — Classifier Fine-tuning (full imbalanced train set)')
print('=' * 60)

# Unfreeze GloVe embeddings for Stage 2 — BCE signal is cleaner than SupCon
for layer in classifier.layers:
    if isinstance(layer, tf.keras.layers.Embedding):
        layer.trainable = True
        print(f'  Embedding unfrozen ({layer.name}) for Stage 2')
        break

# Balanced batch generator — 50% genuine / 50% fake per batch, no augmentation.
# Balanced batches align training with the macro F1 evaluation objective (50/50
# balanced test set) and avoid the instability of extreme class weights (~4.5x).
steps_per_epoch_clf = len(X_train) // BATCH_CLF
clf_gen = balanced_batch_gen(
    X_train_seq, y_train_arr, BATCH_CLF, augment=False
)
print(f'Balanced Stage 2 batches: {BATCH_CLF//2} genuine + {BATCH_CLF//2} fake per batch')
print(f'Steps per epoch: {steps_per_epoch_clf:,}')


# ── Custom callback: early stopping + LR reduction on val macro F1 ──
class ValMacroF1Callback(tf.keras.callbacks.Callback):
    """
    Monitors threshold-tuned validation macro F1 for both:
    - LR reduction (lr_patience epochs without improvement → halve LR)
    - Early stopping with best-weight restoration (es_patience)
    Both use independent counters, both reset on improvement.
    """
    def __init__(self, X_val, y_val, batch_size, es_patience,
                 lr_patience=2, lr_factor=0.5, min_lr=1e-6):
        super().__init__()
        self.X_val          = X_val
        self.y_val          = y_val
        self.batch_size     = batch_size
        self.es_patience    = es_patience
        self.lr_patience    = lr_patience
        self.lr_factor      = lr_factor
        self.min_lr         = min_lr
        self.best_f1        = 0.0
        self.best_threshold = 0.5
        self.best_weights   = None
        self.es_wait        = 0
        self.lr_wait        = 0

    def _get_lr(self):
        return float(self.model.optimizer.learning_rate)

    def _set_lr(self, new_lr):
        # Direct assignment works in Keras 3 (TF 2.16+) where
        # optimizer.learning_rate is a plain float, not a tf.Variable.
        try:
            self.model.optimizer.learning_rate.assign(new_lr)
        except AttributeError:
            self.model.optimizer.learning_rate = new_lr

    def on_epoch_end(self, epoch, logs=None):
        probs = self.model.predict(
            self.X_val, batch_size=self.batch_size, verbose=0
        ).flatten()
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.10, 0.90, 0.01):
            f = f1_score(
                self.y_val, (probs >= t).astype(int),
                average='macro', zero_division=0,
            )
            if f > best_f1:
                best_f1, best_t = f, round(float(t), 2)

        print(f'  Val Macro F1 (tuned): {best_f1:.4f}  threshold: {best_t:.2f}', end='')

        if best_f1 > self.best_f1:
            self.best_f1        = best_f1
            self.best_threshold = best_t
            self.best_weights   = self.model.get_weights()
            self.es_wait        = 0
            self.lr_wait        = 0
            print('  [improved]')
        else:
            self.es_wait += 1
            self.lr_wait += 1
            # LR reduction check
            if self.lr_wait >= self.lr_patience:
                old_lr = self._get_lr()
                new_lr = max(old_lr * self.lr_factor, self.min_lr)
                if new_lr < old_lr:
                    self._set_lr(new_lr)
                    print(f'  [no improvement, es {self.es_wait}/{self.es_patience}'
                          f'  LR {old_lr:.2e} -> {new_lr:.2e}]')
                else:
                    print(f'  [no improvement, es {self.es_wait}/{self.es_patience}  LR at min]')
                self.lr_wait = 0
            else:
                print(f'  [no improvement, es {self.es_wait}/{self.es_patience}]')
            # Early stopping check
            if self.es_wait >= self.es_patience:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)
            print(
                f'\nBest weights restored — '
                f'Val Macro F1: {self.best_f1:.4f}  '
                f'threshold: {self.best_threshold:.2f}'
            )


# ── Compile and train ─────────────────────────────────────────────
# Re-compile after changing embedding trainable state.
classifier.compile(
    optimizer=Adam(learning_rate=LR_CLF),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

f1_cb = ValMacroF1Callback(
    X_val_seq, y_val_arr, BATCH_CLF, PATIENCE_CLF,
    lr_patience=2, lr_factor=0.5, min_lr=1e-6,
)

clf_history = classifier.fit(
    clf_gen,
    steps_per_epoch=steps_per_epoch_clf,
    validation_data=(X_val_seq, y_val_arr),
    epochs=EPOCHS_CLF,
    callbacks=[f1_cb],
)

In [ ]:
# ── Threshold and best val F1 come from the callback ─────────────
best_thresh = f1_cb.best_threshold
best_val_f1 = f1_cb.best_f1
print(f'Best threshold : {best_thresh:.2f}  (Val macro F1: {best_val_f1:.1%})')

# ── Internal test evaluation (imbalanced test set) ────────────────
test_probs = classifier.predict(X_test_seq, batch_size=BATCH_CLF).flatten()
y_pred     = (test_probs >= best_thresh).astype(int)
print(f'\nInternal test results (threshold={best_thresh:.2f}) — imbalanced test set:')
print(classification_report(y_test_arr, y_pred, target_names=['Genuine', 'Fake']))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(range(1, EPOCHS_SUPCON + 1), supcon_history, marker='o')
axes[0].set_title('Stage 1 — SupCon Loss'); axes[0].set_xlabel('Epoch')

axes[1].plot(clf_history.history['accuracy'],     label='Train')
axes[1].plot(clf_history.history['val_accuracy'], label='Val')
axes[1].set_title('Stage 2 — Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(clf_history.history['loss'],     label='Train')
axes[2].plot(clf_history.history['val_loss'], label='Val')
axes[2].set_title('Stage 2 — Loss'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Save metrics CSV ─────────────────────────────────────────────
metrics = {
    'model':                'lstm_contrastive',
    'threshold':            best_thresh,
    'val_macro_f1':         round(best_val_f1, 4),
    'test_accuracy':        round(accuracy_score(y_test_arr, y_pred), 4),
    'test_macro_f1':        round(f1_score(y_test_arr, y_pred, average='macro'), 4),
    'test_fake_precision':  round(precision_score(y_test_arr, y_pred, pos_label=1, zero_division=0), 4),
    'test_fake_recall':     round(recall_score(y_test_arr, y_pred, pos_label=1, zero_division=0), 4),
    'test_fake_f1':         round(f1_score(y_test_arr, y_pred, pos_label=1, zero_division=0), 4),
    'test_genuine_precision': round(precision_score(y_test_arr, y_pred, pos_label=0, zero_division=0), 4),
    'test_genuine_recall':  round(recall_score(y_test_arr, y_pred, pos_label=0, zero_division=0), 4),
    'test_genuine_f1':      round(f1_score(y_test_arr, y_pred, pos_label=0, zero_division=0), 4),
}
pd.DataFrame([metrics]).to_csv('yelp_fake_lstm_contrastive_metrics.csv', index=False)
print('Metrics saved: yelp_fake_lstm_contrastive_metrics.csv')

# ── Save config JSON ─────────────────────────────────────────────
config = {
    'max_len': MAX_LEN, 'vocab_size': VOCAB_SIZE, 'embed_dim': EMBED_DIM,
    'lstm_units': LSTM_UNITS, 'proj_dim_1': PROJ_DIM_1, 'proj_dim_2': PROJ_DIM_2,
    'supcon_temp': SUPCON_TEMP, 'batch_supcon': BATCH_SUPCON,
    'epochs_supcon': EPOCHS_SUPCON, 'lr_supcon': LR_SUPCON,
    'augment_drop_rate': AUGMENT_DROP_RATE,
    'batch_clf': BATCH_CLF, 'epochs_clf': EPOCHS_CLF,
    'lr_clf': LR_CLF, 'patience_clf': PATIENCE_CLF,
}
with open('yelp_fake_lstm_contrastive_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Config saved: yelp_fake_lstm_contrastive_config.json')

# ── Save tokenizer + threshold ───────────────────────────────────
with open('yelp_fake_lstm_contrastive_tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(tokenizer.to_json())
print('Tokenizer saved: yelp_fake_lstm_contrastive_tokenizer.json')

with open('yelp_fake_lstm_contrastive_threshold.json', 'w') as f:
    json.dump({'threshold': best_thresh}, f, indent=2)
print(f'Threshold saved: yelp_fake_lstm_contrastive_threshold.json  (value: {best_thresh})')

classifier.save('yelp_fake_lstm_contrastive.keras')
print('Keras model saved: yelp_fake_lstm_contrastive.keras')

In [ ]:
import subprocess

cpu_script = """\
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import tensorflow as tf
model = tf.keras.models.load_model('yelp_fake_lstm_contrastive.keras', compile=False)
model.export('yelp_fake_lstm_contrastive_savedmodel')
print('CPU SavedModel export done')
"""
with open('cpu_export.py', 'w') as f:
    f.write(cpu_script)

r1 = subprocess.run(['python', 'cpu_export.py'], capture_output=True, text=True,
                    env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''})
print(r1.stdout)
if r1.returncode != 0:
    print('STDERR:', r1.stderr)

r2 = subprocess.run(
    ['python', '-m', 'tf2onnx.convert',
     '--saved-model', 'yelp_fake_lstm_contrastive_savedmodel',
     '--output',      'yelp_fake_lstm_contrastive.onnx',
     '--opset', '13'],
    capture_output=True, text=True
)
print(r2.stdout)
if r2.returncode != 0:
    print('STDERR:', r2.stderr)

In [ ]:
import onnxruntime as ort

sess       = ort.InferenceSession('yelp_fake_lstm_contrastive.onnx', providers=['CPUExecutionProvider'])
input_name = sess.get_inputs()[0].name
print('ONNX inputs :', [i.name for i in sess.get_inputs()])
print('ONNX outputs:', [o.name for o in sess.get_outputs()])

sample = X_test_seq[:4].astype(np.float32)
preds  = sess.run(None, {input_name: sample})[0]
print('Sample predictions:', preds.flatten())

In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile('yelp_fake_lstm_contrastive_onnx.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('yelp_fake_lstm_contrastive.onnx')
    zf.write('yelp_fake_lstm_contrastive_tokenizer.json')
    zf.write('yelp_fake_lstm_contrastive_threshold.json')
    zf.write('yelp_fake_lstm_contrastive_metrics.csv')
    zf.write('yelp_fake_lstm_contrastive_config.json')
print('Created: yelp_fake_lstm_contrastive_onnx.zip')

files.download('yelp_fake_lstm_contrastive_onnx.zip')

## Next steps
Extract `yelp_fake_lstm_contrastive_onnx.zip` and place the folder at:
```
data/yelp_fake_lstm_contrastive_onnx/
    yelp_fake_lstm_contrastive.onnx
    yelp_fake_lstm_contrastive_tokenizer.json
    yelp_fake_lstm_contrastive_threshold.json
    yelp_fake_lstm_contrastive_metrics.csv
    yelp_fake_lstm_contrastive_config.json
```
Then run `evaluate_yelp_ablation.py` to compare all six models on the balanced test set.